In [1]:
from pathlib import Path
import shutil

REPO = Path(
    "/home/sagemaker-user/"
    "Beverage-Price-Prediction-AWS"
)

DEPLOY_DIR = REPO / "src" / "deployment"

if DEPLOY_DIR.exists():
    shutil.rmtree(DEPLOY_DIR)

(DEPLOY_DIR / "preprocessing").mkdir(
    parents=True,
    exist_ok=True
)

print(DEPLOY_DIR)

/home/sagemaker-user/Beverage-Price-Prediction-AWS/src/deployment


In [2]:
shutil.copy(
    REPO / "src/inference/inference.py",
    DEPLOY_DIR / "inference.py"
)

shutil.copy(
    REPO / "src/inference/requirements.txt",
    DEPLOY_DIR / "requirements.txt"
)

shutil.copy(
    REPO / "src/preprocessing/feature_utils.py",
    DEPLOY_DIR / "preprocessing/feature_utils.py"
)

shutil.copy(
    REPO / "src/preprocessing/__init__.py",
    DEPLOY_DIR / "preprocessing/__init__.py"
)

print("Deployment files:")

for path in DEPLOY_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(DEPLOY_DIR))

Deployment files:
inference.py
requirements.txt
preprocessing/feature_utils.py
preprocessing/__init__.py


In [5]:
import boto3

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)

from sagemaker.core.image_uris import retrieve
from sagemaker.core.utils import repack_model


AWS_REGION = "ap-south-1"

MODEL_PACKAGE_GROUP_NAME = (
    "beverage-price-prediction-xgboost"
)

MODEL_ARTIFACT_URI = (
    "s3://krushang-beverage-ml-2026/models/"
    "beverage-xgboost-production-20260904074049/"
    "output/model.tar.gz"
)

EVALUATION_URI = (
    "s3://krushang-beverage-ml-2026/"
    "evaluation/xgboost_final_test_metrics.json"
)


sm = boto3.client(
    "sagemaker",
    region_name=AWS_REGION
)

sagemaker_session = Session()

role = get_execution_role()


print("Region:", sagemaker_session.boto_region_name)
print("Role:", role)
print("Model artifact:", MODEL_ARTIFACT_URI)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: ap-south-1
Role: arn:aws:iam::812224290846:role/service-role/AmazonSageMaker-ExecutionRole-20260902T184223
Model artifact: s3://krushang-beverage-ml-2026/models/beverage-xgboost-production-20260904074049/output/model.tar.gz


In [4]:
try:
    response = sm.create_model_package_group(
        ModelPackageGroupName=
            MODEL_PACKAGE_GROUP_NAME,

        ModelPackageGroupDescription=(
            "Versioned XGBoost models for "
            "beverage price range prediction"
        )
    )

    print(
        "Created:",
        response["ModelPackageGroupArn"]
    )

except sm.exceptions.ClientError as e:

    if "already exists" in str(e).lower():
        print(
            "Model Package Group already exists"
        )
    else:
        raise

Created: arn:aws:sagemaker:ap-south-1:812224290846:model-package-group/beverage-price-prediction-xgboost


In [6]:
REPACKED_MODEL_URI = (
    "s3://krushang-beverage-ml-2026/"
    "models/deployment/"
    "beverage-xgboost-v1/"
    "model.tar.gz"
)

repack_model(
    inference_script="inference.py",
    source_directory=str(DEPLOY_DIR),
    dependencies=[],
    model_uri=MODEL_ARTIFACT_URI,
    repacked_model_uri=REPACKED_MODEL_URI,
    sagemaker_session=sagemaker_session,
)

print("✅ Model repacked")
print("Deployment artifact:", REPACKED_MODEL_URI)

✅ Model repacked
Deployment artifact: s3://krushang-beverage-ml-2026/models/deployment/beverage-xgboost-v1/model.tar.gz


In [8]:
INFERENCE_IMAGE_URI = retrieve(
    framework="sklearn",
    region=AWS_REGION,
    version="1.4-2-py312",
    image_scope="inference",
)

print("Inference image:")
print(INFERENCE_IMAGE_URI)

[09/04/26 09:23:56] INFO     Defaulting to only available Python version: py3                     ]8;id=9254183;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=9254184;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Defaulting to only supported image scope: cpu.                       ]8;id=9254190;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=9254191;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#539\539]8;;\

Inference image:
720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3


In [9]:
model_metrics = {
    "ModelQuality": {
        "Statistics": {
            "ContentType": "application/json",
            "S3Uri": EVALUATION_URI
        }
    }
}

response = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,

    ModelPackageDescription=(
        "XGBoost production model for beverage "
        "price-range prediction"
    ),

    InferenceSpecification={
        "Containers": [
            {
                "Image": INFERENCE_IMAGE_URI,
                "ModelDataUrl": REPACKED_MODEL_URI,

                "Environment": {
                    "SAGEMAKER_PROGRAM": "inference.py",
                    "SAGEMAKER_SUBMIT_DIRECTORY":
                        "/opt/ml/model/code"
                }
            }
        ],

        "SupportedContentTypes": [
            "application/json"
        ],

        "SupportedResponseMIMETypes": [
            "application/json"
        ],
    },

    ModelApprovalStatus="PendingManualApproval",

    ModelMetrics=model_metrics
)

MODEL_PACKAGE_ARN = response["ModelPackageArn"]

print("✅ Model Version registered")
print("Model Package ARN:", MODEL_PACKAGE_ARN)

✅ Model Version registered
Model Package ARN: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/1


In [10]:
package = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)

print("Version:", package["ModelPackageVersion"])
print("Status:", package["ModelPackageStatus"])
print("Approval:", package["ModelApprovalStatus"])
print(
    "Model artifact:",
    package["InferenceSpecification"]
           ["Containers"][0]
           ["ModelDataUrl"]
)

Version: 1
Status: Completed
Approval: PendingManualApproval
Model artifact: s3://krushang-beverage-ml-2026/models/deployment/beverage-xgboost-v1/model.tar.gz


In [11]:
package = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)

print("Version:", package["ModelPackageVersion"])
print("Status:", package["ModelPackageStatus"])
print("Approval:", package["ModelApprovalStatus"])
print(
    "Artifact:",
    package["InferenceSpecification"]
           ["Containers"][0]
           ["ModelDataUrl"]
)

Version: 1
Status: Completed
Approval: Approved
Artifact: s3://krushang-beverage-ml-2026/models/deployment/beverage-xgboost-v1/model.tar.gz


## Model Version 2 — Serverless Inference Compatibility Fix

Version 1 failed endpoint startup because the Python 3.12 inference
container blocked runtime pip installation under PEP 668.

Version 2:
- uses corrected inference import path
- enables pip installation in the managed inference container
- preserves the original trained model

In [1]:
from pathlib import Path
import shutil

REPO = Path(
    "/home/sagemaker-user/"
    "Beverage-Price-Prediction-AWS"
)

DEPLOY_DIR = REPO / "src" / "deployment"

# Recreate deployment source cleanly
if DEPLOY_DIR.exists():
    shutil.rmtree(DEPLOY_DIR)

(DEPLOY_DIR / "preprocessing").mkdir(
    parents=True,
    exist_ok=True
)

# Copy corrected inference code
shutil.copy(
    REPO / "src/inference/inference.py",
    DEPLOY_DIR / "inference.py"
)

shutil.copy(
    REPO / "src/inference/requirements.txt",
    DEPLOY_DIR / "requirements.txt"
)

shutil.copy(
    REPO / "src/preprocessing/feature_utils.py",
    DEPLOY_DIR / "preprocessing/feature_utils.py"
)

shutil.copy(
    REPO / "src/preprocessing/__init__.py",
    DEPLOY_DIR / "preprocessing/__init__.py"
)

print("Deployment files:")

for path in DEPLOY_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(DEPLOY_DIR))

Deployment files:
inference.py
requirements.txt
preprocessing/feature_utils.py
preprocessing/__init__.py


In [2]:
print(
    (DEPLOY_DIR / "inference.py").read_text()[
        :1200
    ]
)


import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# --------------------------------------------------
# Allow access to sibling preprocessing package
# --------------------------------------------------
CODE_DIR = Path(__file__).resolve().parent

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from preprocessing.feature_utils import transform_features


# --------------------------------------------------
# Load model
# --------------------------------------------------

def model_fn(model_dir):

    bundle_path = os.path.join(
        model_dir,
        "model_bundle.joblib"
    )

    bundle = joblib.load(bundle_path)

    return bundle


# --------------------------------------------------
# Deserialize request
# --------------------------------------------------

def input_fn(request_body, request_content_type):

    if request_content_type != "application/json":
        raise ValueError(
   

In [3]:
import boto3

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)

from sagemaker.core.image_uris import retrieve
from sagemaker.core.utils import repack_model


AWS_REGION = "ap-south-1"

MODEL_PACKAGE_GROUP_NAME = (
    "beverage-price-prediction-xgboost"
)

MODEL_ARTIFACT_URI = (
    "s3://krushang-beverage-ml-2026/models/"
    "beverage-xgboost-production-20260904074049/"
    "output/model.tar.gz"
)

EVALUATION_URI = (
    "s3://krushang-beverage-ml-2026/"
    "evaluation/xgboost_final_test_metrics.json"
)

REPACKED_MODEL_URI_V2 = (
    "s3://krushang-beverage-ml-2026/"
    "models/deployment/"
    "beverage-xgboost-v2/"
    "model.tar.gz"
)


sm = boto3.client(
    "sagemaker",
    region_name=AWS_REGION
)

sagemaker_session = Session()

role = get_execution_role()


INFERENCE_IMAGE_URI = retrieve(
    framework="sklearn",
    region=AWS_REGION,
    version="1.4-2-py312",
    image_scope="inference",
)


print("Inference image:")
print(INFERENCE_IMAGE_URI)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[09/04/26 16:07:55] INFO     Defaulting to only available Python version: py3                     ]8;id=15461275;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=15461276;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Defaulting to only supported image scope: cpu.                       ]8;id=15461282;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=15461283;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#539\539]8;;\

Inference image:
720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3


In [4]:
repack_model(
    inference_script="inference.py",
    source_directory=str(DEPLOY_DIR),
    dependencies=[],
    model_uri=MODEL_ARTIFACT_URI,
    repacked_model_uri=REPACKED_MODEL_URI_V2,
    sagemaker_session=sagemaker_session,
)

print("✅ V2 deployment artifact created")
print(REPACKED_MODEL_URI_V2)

✅ V2 deployment artifact created
s3://krushang-beverage-ml-2026/models/deployment/beverage-xgboost-v2/model.tar.gz


In [5]:
model_metrics = {
    "ModelQuality": {
        "Statistics": {
            "ContentType": "application/json",
            "S3Uri": EVALUATION_URI
        }
    }
}


response = sm.create_model_package(
    ModelPackageGroupName=
        MODEL_PACKAGE_GROUP_NAME,

    ModelPackageDescription=(
        "XGBoost production model V2 - "
        "serverless inference compatibility fix"
    ),

    InferenceSpecification={
        "Containers": [
            {
                "Image":
                    INFERENCE_IMAGE_URI,

                "ModelDataUrl":
                    REPACKED_MODEL_URI_V2,

                "Environment": {
                    "SAGEMAKER_PROGRAM":
                        "inference.py",

                    "SAGEMAKER_SUBMIT_DIRECTORY":
                        "/opt/ml/model/code",

                    "PIP_BREAK_SYSTEM_PACKAGES":
                        "1"
                }
            }
        ],

        "SupportedContentTypes": [
            "application/json"
        ],

        "SupportedResponseMIMETypes": [
            "application/json"
        ],
    },

    ModelApprovalStatus=
        "PendingManualApproval",

    ModelMetrics=model_metrics
)


MODEL_PACKAGE_ARN_V2 = (
    response["ModelPackageArn"]
)

print("✅ Model Version 2 registered")
print(MODEL_PACKAGE_ARN_V2)

✅ Model Version 2 registered
arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/2


In [6]:
package_v2 = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN_V2
)

print(
    "Version:",
    package_v2["ModelPackageVersion"]
)

print(
    "Status:",
    package_v2["ModelPackageStatus"]
)

print(
    "Approval:",
    package_v2["ModelApprovalStatus"]
)

container = (
    package_v2[
        "InferenceSpecification"
    ]["Containers"][0]
)

print(
    "Artifact:",
    container["ModelDataUrl"]
)

print(
    "PIP override:",
    container["Environment"].get(
        "PIP_BREAK_SYSTEM_PACKAGES"
    )
)

Version: 2
Status: Completed
Approval: PendingManualApproval
Artifact: s3://krushang-beverage-ml-2026/models/deployment/beverage-xgboost-v2/model.tar.gz
PIP override: 1


In [7]:
packages = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Ascending"
)

for pkg in packages["ModelPackageSummaryList"]:
    print(
        "Version:", pkg["ModelPackageVersion"],
        "| Approval:", pkg["ModelApprovalStatus"],
        "| ARN:", pkg["ModelPackageArn"]
    )

Version: 1 | Approval: Approved | ARN: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/1
Version: 2 | Approval: PendingManualApproval | ARN: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/2


In [8]:
MODEL_PACKAGE_ARN_V1 = next(
    pkg["ModelPackageArn"]
    for pkg in packages["ModelPackageSummaryList"]
    if pkg["ModelPackageVersion"] == 1
)

sm.update_model_package(
    ModelPackageArn=MODEL_PACKAGE_ARN_V1,
    ModelApprovalStatus="Rejected",
    ApprovalDescription=(
        "Deployment validation failed. "
        "Inference container could not install runtime "
        "dependencies under Python 3.12 PEP 668."
    )
)

print("✅ Version 1 marked Rejected")

✅ Version 1 marked Rejected


In [9]:
sm.update_model_package(
    ModelPackageArn=MODEL_PACKAGE_ARN_V2,
    ModelApprovalStatus="Approved",
    ApprovalDescription=(
        "Approved after serverless inference compatibility fixes."
    )
)

print("✅ Version 2 approved")

✅ Version 2 approved


In [10]:
for arn in [
    MODEL_PACKAGE_ARN_V1,
    MODEL_PACKAGE_ARN_V2
]:
    p = sm.describe_model_package(
        ModelPackageName=arn
    )

    print(
        "Version:", p["ModelPackageVersion"],
        "| Approval:", p["ModelApprovalStatus"],
        "| Status:", p["ModelPackageStatus"]
    )

Version: 1 | Approval: Rejected | Status: Completed
Version: 2 | Approval: Approved | Status: Completed


## Model Version 3 — Explicit Inference Packaging

Fixes:
- explicit setup.py for inference module installation
- CPU-only XGBoost dependency
- preserves corrected runtime import path

In [12]:
from pathlib import Path
import shutil

REPO = Path(
    "/home/sagemaker-user/"
    "Beverage-Price-Prediction-AWS"
)

DEPLOY_DIR = REPO / "src" / "deployment"

if DEPLOY_DIR.exists():
    shutil.rmtree(DEPLOY_DIR)

(DEPLOY_DIR / "preprocessing").mkdir(
    parents=True,
    exist_ok=True
)


# Main inference module
shutil.copy(
    REPO / "src/inference/inference.py",
    DEPLOY_DIR / "inference.py"
)

# Explicit Python package definition
shutil.copy(
    REPO / "src/inference/setup.py",
    DEPLOY_DIR / "setup.py"
)

# Runtime dependencies
shutil.copy(
    REPO / "src/inference/requirements.txt",
    DEPLOY_DIR / "requirements.txt"
)

# Shared preprocessing code
shutil.copy(
    REPO / "src/preprocessing/feature_utils.py",
    DEPLOY_DIR / "preprocessing/feature_utils.py"
)

shutil.copy(
    REPO / "src/preprocessing/__init__.py",
    DEPLOY_DIR / "preprocessing/__init__.py"
)


print("V3 deployment source:")

for path in sorted(DEPLOY_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(DEPLOY_DIR))

V3 deployment source:
inference.py
preprocessing/__init__.py
preprocessing/feature_utils.py
requirements.txt
setup.py


In [13]:
print("REQUIREMENTS")
print("-" * 50)
print(
    (DEPLOY_DIR / "requirements.txt").read_text()
)

print("\nSETUP.PY")
print("-" * 50)
print(
    (DEPLOY_DIR / "setup.py").read_text()
)

print("\nINFERENCE PATH CHECK")
print("-" * 50)

inference_text = (
    DEPLOY_DIR / "inference.py"
).read_text()

print(
    "Correct CODE_DIR:",
    "Path(__file__).resolve().parent"
    in inference_text
)

print(
    "Old parents[1] absent:",
    "Path(__file__).resolve().parents[1]"
    not in inference_text
)

REQUIREMENTS
--------------------------------------------------
xgboost-cpu==2.1.4
joblib>=1.3,<2


SETUP.PY
--------------------------------------------------
from setuptools import setup, find_packages

setup(
    name="beverage-inference",
    version="1.0.0",
    py_modules=["inference"],
    packages=find_packages(),
)


INFERENCE PATH CHECK
--------------------------------------------------
Correct CODE_DIR: True
Old parents[1] absent: True


In [14]:
REPACKED_MODEL_URI_V3 = (
    "s3://krushang-beverage-ml-2026/"
    "models/deployment/"
    "beverage-xgboost-v3/"
    "model.tar.gz"
)

repack_model(
    inference_script="inference.py",
    source_directory=str(DEPLOY_DIR),
    dependencies=[],
    model_uri=MODEL_ARTIFACT_URI,
    repacked_model_uri=REPACKED_MODEL_URI_V3,
    sagemaker_session=sagemaker_session,
)

print("✅ V3 artifact repacked")
print(REPACKED_MODEL_URI_V3)

✅ V3 artifact repacked
s3://krushang-beverage-ml-2026/models/deployment/beverage-xgboost-v3/model.tar.gz


In [15]:
import boto3
import io
import tarfile
from urllib.parse import urlparse

s3 = boto3.client(
    "s3",
    region_name=AWS_REGION
)

parsed = urlparse(
    REPACKED_MODEL_URI_V3
)

bucket = parsed.netloc
key = parsed.path.lstrip("/")

artifact_bytes = s3.get_object(
    Bucket=bucket,
    Key=key
)["Body"].read()


with tarfile.open(
    fileobj=io.BytesIO(artifact_bytes),
    mode="r:gz"
) as tar:

    names = sorted(tar.getnames())


print("V3 model.tar.gz contents:\n")

for name in names:
    print(name)

V3 model.tar.gz contents:


code
code/inference.py
code/preprocessing
code/preprocessing/__init__.py
code/preprocessing/feature_utils.py
code/requirements.txt
code/setup.py
model_bundle.joblib
training_metadata.json
xgboost_model.json


In [16]:
required_files = {
    "model_bundle.joblib",
    "training_metadata.json",
    "xgboost_model.json",
    "code/inference.py",
    "code/requirements.txt",
    "code/setup.py",
    "code/preprocessing/__init__.py",
    "code/preprocessing/feature_utils.py",
}

missing = required_files - set(names)

if missing:
    print("❌ Missing files:")
    for file in sorted(missing):
        print(file)
else:
    print(
        "✅ V3 deployment artifact structure validated"
    )

✅ V3 deployment artifact structure validated


### Register Model Version 3

In [17]:
model_metrics = {
    "ModelQuality": {
        "Statistics": {
            "ContentType": "application/json",
            "S3Uri": EVALUATION_URI
        }
    }
}

response = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,

    ModelPackageDescription=(
        "XGBoost production model V3 - "
        "explicit inference packaging and CPU-only runtime"
    ),

    InferenceSpecification={
        "Containers": [
            {
                "Image": INFERENCE_IMAGE_URI,

                "ModelDataUrl": REPACKED_MODEL_URI_V3,

                "Environment": {
                    "SAGEMAKER_PROGRAM":
                        "inference.py",

                    "SAGEMAKER_SUBMIT_DIRECTORY":
                        "/opt/ml/model/code",

                    "PIP_BREAK_SYSTEM_PACKAGES":
                        "1"
                }
            }
        ],

        "SupportedContentTypes": [
            "application/json"
        ],

        "SupportedResponseMIMETypes": [
            "application/json"
        ]
    },

    ModelApprovalStatus=
        "PendingManualApproval",

    ModelMetrics=model_metrics
)

MODEL_PACKAGE_ARN_V3 = (
    response["ModelPackageArn"]
)

print("✅ Model Version 3 registered")
print(MODEL_PACKAGE_ARN_V3)

✅ Model Version 3 registered
arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/3


In [18]:
package_v3 = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN_V3
)

container_v3 = (
    package_v3[
        "InferenceSpecification"
    ]["Containers"][0]
)

print(
    "Version:",
    package_v3["ModelPackageVersion"]
)

print(
    "Status:",
    package_v3["ModelPackageStatus"]
)

print(
    "Approval:",
    package_v3["ModelApprovalStatus"]
)

print(
    "Artifact:",
    container_v3["ModelDataUrl"]
)

print(
    "Image:",
    container_v3["Image"]
)

print(
    "PIP override:",
    container_v3["Environment"].get(
        "PIP_BREAK_SYSTEM_PACKAGES"
    )
)

Version: 3
Status: Completed
Approval: PendingManualApproval
Artifact: s3://krushang-beverage-ml-2026/models/deployment/beverage-xgboost-v3/model.tar.gz
Image: 720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3
PIP override: 1


### Governance Update — Reject V2 and Approve V3

In [19]:
packages = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Ascending"
)

package_by_version = {
    pkg["ModelPackageVersion"]: pkg["ModelPackageArn"]
    for pkg in packages["ModelPackageSummaryList"]
}

MODEL_PACKAGE_ARN_V2 = package_by_version[2]
MODEL_PACKAGE_ARN_V3 = package_by_version[3]

print("V2:", MODEL_PACKAGE_ARN_V2)
print("V3:", MODEL_PACKAGE_ARN_V3)

V2: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/2
V3: arn:aws:sagemaker:ap-south-1:812224290846:model-package/beverage-price-prediction-xgboost/3


In [20]:
sm.update_model_package(
    ModelPackageArn=MODEL_PACKAGE_ARN_V2,
    ModelApprovalStatus="Rejected",
    ApprovalDescription=(
        "Deployment validation failed because the "
        "inference module was not importable in the "
        "SageMaker serving container."
    )
)

print("✅ Version 2 rejected")

✅ Version 2 rejected


In [21]:
sm.update_model_package(
    ModelPackageArn=MODEL_PACKAGE_ARN_V3,
    ModelApprovalStatus="Approved",
    ApprovalDescription=(
        "Approved after validating explicit inference "
        "packaging, corrected runtime imports, and "
        "CPU-only XGBoost dependencies."
    )
)

print("✅ Version 3 approved")

✅ Version 3 approved


In [22]:
packages = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Ascending"
)

for pkg in packages["ModelPackageSummaryList"]:
    print(
        "Version:",
        pkg["ModelPackageVersion"],
        "| Status:",
        pkg["ModelPackageStatus"],
        "| Approval:",
        pkg["ModelApprovalStatus"]
    )

Version: 1 | Status: Completed | Approval: Rejected
Version: 2 | Status: Completed | Approval: Rejected
Version: 3 | Status: Completed | Approval: Approved
